# NexaTel Customer Churn Analytics
## Phase 3 — Churn KPI Calculations

### Objective

The objective of Phase 3 is to calculate the key business KPIs used by management to evaluate customer retention, revenue, customer value, service quality, and contract renewals.

### KPIs

1. Customer Churn Rate
2. Customer Retention Rate
3. ARPU
4. Customer Lifetime Value
5. Revenue Lost to Churn
6. Monthly Recurring Revenue
7. First Contact Resolution
8. Average Resolution Time
9. Contract Renewal Rate
10. Average Customer Tenure

### Bonus Analysis

- Cohort Retention Analysis at 3, 6 and 12 months

### Import Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


### Data Folder

In [3]:
DATA_DIR = Path(r"C:\Users\ASUS\Desktop\Internmo\Project - 2")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data folder not found:\n{DATA_DIR}\n\n"
        "Please check the DATA_DIR path."
    )

print("Data folder found:")
print(DATA_DIR)

Data folder found:
C:\Users\ASUS\Desktop\Internmo\Project - 2


### All CSV Files

In [4]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")
print()

for file in csv_files:
    print(file.name)

CSV files found: 24

billing.csv
cities.csv
complaints.csv
contracts.csv
customer_feedback.csv
customers.csv
data_quality_issue_log.csv
devices.csv
employees.csv
marketing_campaigns.csv
network_quality.csv
payments.csv
plan_history.csv
plans.csv
recharges.csv
regions.csv
retention_campaigns.csv
states.csv
stores.csv
subscriptions.csv
support_tickets.csv
usage_data.csv
usage_sms.csv
usage_voice.csv


### Load All Tables

In [5]:
data = {}

for file in csv_files:
    table_name = file.stem
    data[table_name] = pd.read_csv(file)

print(f"Successfully loaded {len(data)} tables.")

Successfully loaded 24 tables.


### Check Loaded Tables

In [7]:
for table_name, df in data.items():
    print(f"{table_name:30} {df.shape}")

billing                        (115313, 9)
cities                         (68, 8)
complaints                     (16584, 7)
contracts                      (10607, 9)
customer_feedback              (28499, 7)
customers                      (19076, 25)
data_quality_issue_log         (20, 3)
devices                        (19000, 8)
employees                      (800, 9)
marketing_campaigns            (40, 9)
network_quality                (57000, 8)
payments                       (92362, 7)
plan_history                   (19349, 6)
plans                          (25, 10)
recharges                      (146501, 6)
regions                        (4, 3)
retention_campaigns            (6475, 8)
states                         (36, 4)
stores                         (2180, 7)
subscriptions                  (20523, 8)
support_tickets                (31490, 11)
usage_data                     (57000, 5)
usage_sms                      (57000, 5)
usage_voice                    (57000, 6)


### Extract Required Tables

In [8]:
customers = data["customers"].copy()
subscriptions = data["subscriptions"].copy()
plans = data["plans"].copy()
billing = data["billing"].copy()
support_tickets = data["support_tickets"].copy()
contracts = data["contracts"].copy()

print("Required Phase 3 tables loaded successfully.")

Required Phase 3 tables loaded successfully.


### Check Required Columns

In [9]:
required_columns = {
    "customers": [
        "customer_id",
        "customer_status",
        "acquisition_date",
        "churn_date",
        "tenure_months",
        "customer_segment",
        "plan_id"
    ],
    
    "subscriptions": [
        "subscription_id",
        "customer_id",
        "start_date",
        "end_date",
        "billing_cycle",
        "monthly_charge_inr",
        "subscription_status"
    ],
    
    "plans": [
        "plan_id",
        "monthly_charge"
    ],
    
    "billing": [
        "invoice_id",
        "customer_id",
        "billing_period_month",
        "total_amount_inr"
    ],
    
    "support_tickets": [
        "ticket_id",
        "customer_id",
        "first_contact_resolution",
        "resolution_hours"
    ],
    
    "contracts": [
        "contract_id",
        "customer_id",
        "end_date",
        "renewal_status",
        "contract_type"
    ]
}

tables = {
    "customers": customers,
    "subscriptions": subscriptions,
    "plans": plans,
    "billing": billing,
    "support_tickets": support_tickets,
    "contracts": contracts
}

for table_name, columns in required_columns.items():
    
    missing = [
        column for column in columns
        if column not in tables[table_name].columns
    ]
    
    if missing:
        print(f"{table_name}: Missing columns -> {missing}")
    else:
        print(f"{table_name}: All required columns available")

customers: All required columns available
subscriptions: All required columns available
plans: All required columns available
billing: All required columns available
support_tickets: All required columns available
contracts: All required columns available


### Remove Exact Duplicate Rows

In [10]:
customers_before = len(customers)

customers = customers.drop_duplicates().copy()

customers_after = len(customers)

print("Customers before removing exact duplicates:", customers_before)
print("Customers after removing exact duplicates :", customers_after)
print("Exact duplicate rows removed             :", customers_before - customers_after)

Customers before removing exact duplicates: 19076
Customers after removing exact duplicates : 19000
Exact duplicate rows removed             : 76


### Convert Date Columns

In [11]:
customers["acquisition_date"] = pd.to_datetime(
    customers["acquisition_date"],
    dayfirst=True,
    errors="coerce"
)

customers["churn_date"] = pd.to_datetime(
    customers["churn_date"],
    dayfirst=True,
    errors="coerce"
)

subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"],
    dayfirst=True,
    errors="coerce"
)

subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"],
    dayfirst=True,
    errors="coerce"
)

contracts["end_date"] = pd.to_datetime(
    contracts["end_date"],
    dayfirst=True,
    errors="coerce"
)

billing["billing_date"] = pd.to_datetime(
    billing["billing_date"],
    dayfirst=True,
    errors="coerce"
)

print("Date conversion completed.")

Date conversion completed.


### Convert Numeric Columns

In [12]:
customers["tenure_months"] = pd.to_numeric(
    customers["tenure_months"],
    errors="coerce"
)

subscriptions["monthly_charge_inr"] = pd.to_numeric(
    subscriptions["monthly_charge_inr"],
    errors="coerce"
)

plans["monthly_charge"] = pd.to_numeric(
    plans["monthly_charge"],
    errors="coerce"
)

billing["total_amount_inr"] = pd.to_numeric(
    billing["total_amount_inr"],
    errors="coerce"
)

support_tickets["resolution_hours"] = pd.to_numeric(
    support_tickets["resolution_hours"],
    errors="coerce"
)

print("Numeric conversion completed.")

Numeric conversion completed.


### Find Reporting Month

In [13]:
billing["billing_period"] = pd.to_datetime(
    billing["billing_period_month"].astype(str),
    format="%m-%Y",
    errors="coerce"
)

if billing["billing_period"].notna().sum() == 0:
    raise ValueError(
        "Could not convert billing_period_month. "
        "Check the billing_period_month format."
    )

REPORT_MONTH = billing["billing_period"].max()

MONTH_START = REPORT_MONTH

MONTH_END = (
    REPORT_MONTH
    + pd.offsets.MonthEnd(1)
)

print("Reporting Month:", REPORT_MONTH.strftime("%B %Y"))
print("Month Start:", MONTH_START.date())
print("Month End:", MONTH_END.date())

Reporting Month: March 2026
Month Start: 2026-03-01
Month End: 2026-03-31


### Create Customer Monthly Status

In [14]:
customer_kpi = customers[
    [
        "customer_id",
        "acquisition_date",
        "churn_date",
        "customer_status",
        "tenure_months",
        "customer_segment",
        "plan_id"
    ]
].copy()

### Customers at Start of Month

In [15]:
customer_kpi["active_at_start"] = (
    (customer_kpi["acquisition_date"] < MONTH_START)
    &
    (
        customer_kpi["churn_date"].isna()
        |
        (customer_kpi["churn_date"] >= MONTH_START)
    )
)

customers_at_start = customer_kpi["active_at_start"].sum()

print("Customers at Start:", f"{customers_at_start:,}")

Customers at Start: 15,376


### New Customers

In [16]:
customer_kpi["new_customer"] = (
    customer_kpi["acquisition_date"].between(
        MONTH_START,
        MONTH_END,
        inclusive="both"
    )
)

new_customers = customer_kpi["new_customer"].sum()

print("New Customers:", f"{new_customers:,}")

New Customers: 3


### Churned Customers

In [18]:
customer_kpi["churned_customer"] = (
    customer_kpi["churn_date"].between(
        MONTH_START,
        MONTH_END,
        inclusive="both"
    )
)

churned_customers = customer_kpi["churned_customer"].sum()

print("Churned Customers:", f"{churned_customers:,}")

Churned Customers: 198


### Customers at End

In [19]:
customer_kpi["active_at_end"] = (
    (customer_kpi["acquisition_date"] <= MONTH_END)
    &
    (
        customer_kpi["churn_date"].isna()
        |
        (customer_kpi["churn_date"] > MONTH_END)
    )
)

customers_at_end = customer_kpi["active_at_end"].sum()

print("Customers at End:", f"{customers_at_end:,}")

Customers at End: 15,185


### Customer Churn Rate

In [20]:
if customers_at_start == 0:
    churn_rate = np.nan
else:
    churn_rate = (
        churned_customers
        / customers_at_start
    ) * 100

print("=" * 60)
print("KPI 1 — CUSTOMER CHURN RATE")
print("=" * 60)

print(f"Customers at Start : {customers_at_start:,}")
print(f"Customers Churned  : {churned_customers:,}")
print(f"Churn Rate         : {churn_rate:.2f}%")
print("Target             : < 2.10% monthly")

KPI 1 — CUSTOMER CHURN RATE
Customers at Start : 15,376
Customers Churned  : 198
Churn Rate         : 1.29%
Target             : < 2.10% monthly


### Customer Retention Rate

In [21]:
if customers_at_start == 0:
    retention_rate = np.nan
else:
    retention_rate = (
        (customers_at_end - new_customers)
        / customers_at_start
    ) * 100

print("=" * 60)
print("KPI 2 — CUSTOMER RETENTION RATE")
print("=" * 60)

print(f"Customers at Start : {customers_at_start:,}")
print(f"New Customers      : {new_customers:,}")
print(f"Customers at End   : {customers_at_end:,}")
print(f"Retention Rate     : {retention_rate:.2f}%")

KPI 2 — CUSTOMER RETENTION RATE
Customers at Start : 15,376
New Customers      : 3
Customers at End   : 15,185
Retention Rate     : 98.74%


### Find Active Subscriptions

In [22]:
active_subscriptions = subscriptions[
    (subscriptions["start_date"] <= MONTH_END)
    &
    (
        subscriptions["end_date"].isna()
        |
        (subscriptions["end_date"] > MONTH_END)
    )
].copy()

print("Active subscriptions:", f"{len(active_subscriptions):,}")

Active subscriptions: 16,965


### Calculate MRR

In [23]:
mrr = active_subscriptions[
    "monthly_charge_inr"
].sum()

print("=" * 60)
print("MONTHLY RECURRING REVENUE")
print("=" * 60)

print(f"MRR: ₹{mrr:,.2f}")

MONTHLY RECURRING REVENUE
MRR: ₹14,971,268.00


### Calculate Average Active Subscribers

In [24]:
average_active_subscribers = (
    customers_at_start + customers_at_end
) / 2

print(
    "Average Active Subscribers:",
    f"{average_active_subscribers:,.2f}"
)

Average Active Subscribers: 15,280.50


### Calculate ARPU

In [25]:
if average_active_subscribers == 0:
    arpu = np.nan
else:
    arpu = (
        mrr
        / average_active_subscribers
    )

print("=" * 60)
print("KPI 3 — ARPU")
print("=" * 60)

print(f"MRR                     : ₹{mrr:,.2f}")
print(f"Average Active Customers: {average_active_subscribers:,.2f}")
print(f"ARPU                    : ₹{arpu:,.2f}")

KPI 3 — ARPU
MRR                     : ₹14,971,268.00
Average Active Customers: 15,280.50
ARPU                    : ₹979.76


### KPI 4 — Customer Lifetime Value

In [26]:
GROSS_MARGIN_PERCENT = None

if GROSS_MARGIN_PERCENT is None:
    clv = np.nan

    print("=" * 60)
    print("KPI 4 — CUSTOMER LIFETIME VALUE")
    print("=" * 60)
    print("CLV cannot be finalized yet.")
    print()
    print("Reason:")
    print("Gross Margin % is not available in the provided dataset.")
    print()
    print("Ask your mentor/company for the approved Gross Margin %.")
    
else:
    gross_margin = GROSS_MARGIN_PERCENT / 100
    
    if churn_rate <= 0:
        clv = np.nan
    else:
        clv = (
            arpu
            * gross_margin
            * (1 / (churn_rate / 100))
        )
    
    print("=" * 60)
    print("KPI 4 — CUSTOMER LIFETIME VALUE")
    print("=" * 60)
    print(f"ARPU          : ₹{arpu:,.2f}")
    print(f"Gross Margin  : {GROSS_MARGIN_PERCENT:.2f}%")
    print(f"Churn Rate    : {churn_rate:.2f}%")
    print(f"CLV           : ₹{clv:,.2f}")

KPI 4 — CUSTOMER LIFETIME VALUE
CLV cannot be finalized yet.

Reason:
Gross Margin % is not available in the provided dataset.

Ask your mentor/company for the approved Gross Margin %.


### KPI 5 — Revenue Lost to Churn

In [27]:
plan_arpu = plans[
    ["plan_id", "monthly_charge"]
].copy()

plan_arpu = plan_arpu.rename(
    columns={
        "monthly_charge": "arpu"
    }
)

customer_arpu = customer_kpi[
    ["customer_id", "plan_id"]
].merge(
    plan_arpu,
    on="plan_id",
    how="left"
)

customer_arpu.head()

,customer_id,plan_id,arpu
0,C0000001,PL002,299
1,C0000002,PL017,1499
2,C0000003,PL021,4166
3,C0000004,PL006,300
4,C0000005,PL003,399


### Calculate Revenue Lost

In [28]:
churned_customer_arpu = customer_arpu[
    customer_kpi["churned_customer"].values
].copy()

revenue_lost_to_churn = (
    churned_customer_arpu["arpu"]
    .sum()
)

print("=" * 60)
print("KPI 5 — REVENUE LOST TO CHURN")
print("=" * 60)

print(
    f"Revenue Lost to Churn: ₹{revenue_lost_to_churn:,.2f}"
)

KPI 5 — REVENUE LOST TO CHURN
Revenue Lost to Churn: ₹122,854.00


### KPI 6 — Monthly Recurring Revenue

In [29]:
print("=" * 60)
print("KPI 6 — MONTHLY RECURRING REVENUE")
print("=" * 60)

print(f"Monthly Recurring Revenue: ₹{mrr:,.2f}")

KPI 6 — MONTHLY RECURRING REVENUE
Monthly Recurring Revenue: ₹14,971,268.00


### KPI 7 — First Contact Resolution

In [30]:
support_tickets["first_contact_resolution"] = (
    support_tickets["first_contact_resolution"]
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_fcr = support_tickets[
    support_tickets["first_contact_resolution"].isin(
        ["yes", "no"]
    )
].copy()

total_fcr_tickets = len(valid_fcr)

resolved_first_contact = (
    valid_fcr["first_contact_resolution"]
    == "yes"
).sum()

print("Valid FCR Tickets:", f"{total_fcr_tickets:,}")
print(
    "Resolved on First Contact:",
    f"{resolved_first_contact:,}"
)

Valid FCR Tickets: 31,490
Resolved on First Contact: 12,726


### Calculate FCR

In [31]:
if total_fcr_tickets == 0:
    fcr_rate = np.nan
else:
    fcr_rate = (
        resolved_first_contact
        / total_fcr_tickets
    ) * 100

print("=" * 60)
print("KPI 7 — FIRST CONTACT RESOLUTION")
print("=" * 60)

print(f"FCR Rate : {fcr_rate:.2f}%")
print("Target   : > 78%")

KPI 7 — FIRST CONTACT RESOLUTION
FCR Rate : 40.41%
Target   : > 78%


### Average Resolution Time

In [32]:
valid_resolution_hours = (
    support_tickets["resolution_hours"]
    .dropna()
)

average_resolution_time = (
    valid_resolution_hours.mean()
)

print("=" * 60)
print("KPI 8 — AVERAGE RESOLUTION TIME")
print("=" * 60)

print(
    f"Average Resolution Time: "
    f"{average_resolution_time:.2f} hours"
)

print("Target: < 24 hours")

KPI 8 — AVERAGE RESOLUTION TIME
Average Resolution Time: 16.04 hours
Target: < 24 hours


### KPI 9 — Contract Renewal Rate

In [33]:
contracts["renewal_status"] = (
    contracts["renewal_status"]
    .astype(str)
    .str.strip()
    .str.lower()
)

contracts_due = contracts[
    contracts["end_date"].between(
        MONTH_START,
        MONTH_END,
        inclusive="both"
    )
].copy()

print(
    "Contracts Due for Renewal:",
    f"{len(contracts_due):,}"
)

Contracts Due for Renewal: 73


### Identify Renewed Contracts

In [34]:
renewed_statuses = [
    "renewed",
    "auto-renewed",
    "auto renewed",
    "auto_renewed"
]

contracts_due["renewed"] = (
    contracts_due["renewal_status"]
    .isin(renewed_statuses)
)

renewed_contracts = (
    contracts_due["renewed"].sum()
)

print(
    "Contracts Renewed:",
    f"{renewed_contracts:,}"
)

Contracts Renewed: 43


### Calculate Overall Renewal Rate

In [35]:
contracts_due_count = len(contracts_due)

if contracts_due_count == 0:
    renewal_rate = np.nan
else:
    renewal_rate = (
        renewed_contracts
        / contracts_due_count
    ) * 100

print("=" * 60)
print("KPI 9 — CONTRACT RENEWAL RATE")
print("=" * 60)

print(f"Contracts Due : {contracts_due_count:,}")
print(f"Renewed       : {renewed_contracts:,}")
print(f"Renewal Rate  : {renewal_rate:.2f}%")

KPI 9 — CONTRACT RENEWAL RATE
Contracts Due : 73
Renewed       : 43
Renewal Rate  : 58.90%


### Postpaid/Fibre Renewal Rate

In [36]:
contracts_due["contract_type_clean"] = (
    contracts_due["contract_type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

postpaid_fibre = contracts_due[
    contracts_due["contract_type_clean"].str.contains(
        "postpaid|fiber|fibre",
        regex=True,
        na=False
    )
].copy()

postpaid_fibre_due = len(postpaid_fibre)

postpaid_fibre_renewed = (
    postpaid_fibre["renewed"].sum()
)

if postpaid_fibre_due == 0:
    postpaid_fibre_renewal_rate = np.nan
else:
    postpaid_fibre_renewal_rate = (
        postpaid_fibre_renewed
        / postpaid_fibre_due
    ) * 100

print("Postpaid/Fibre Contracts Due:",
      f"{postpaid_fibre_due:,}")

print("Postpaid/Fibre Renewed:",
      f"{postpaid_fibre_renewed:,}")

print("Postpaid/Fibre Renewal Rate:",
      f"{postpaid_fibre_renewal_rate:.2f}%")

print("Target: > 88%")

Postpaid/Fibre Contracts Due: 44
Postpaid/Fibre Renewed: 26
Postpaid/Fibre Renewal Rate: 59.09%
Target: > 88%


### Average Customer Tenure

In [37]:
active_customers = customer_kpi[
    customer_kpi["active_at_end"]
].copy()

average_customer_tenure = (
    active_customers["tenure_months"]
    .mean()
)

print("=" * 60)
print("KPI 10 — AVERAGE CUSTOMER TENURE")
print("=" * 60)

print(
    f"Average Customer Tenure: "
    f"{average_customer_tenure:.2f} months"
)

KPI 10 — AVERAGE CUSTOMER TENURE
Average Customer Tenure: 87.10 months


### Final KPI Summary

In [38]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Customer Churn Rate",
        "Customer Retention Rate",
        "ARPU",
        "Customer Lifetime Value",
        "Revenue Lost to Churn",
        "Monthly Recurring Revenue",
        "First Contact Resolution",
        "Average Resolution Time",
        "Contract Renewal Rate",
        "Average Customer Tenure"
    ],
    
    "Value": [
        churn_rate,
        retention_rate,
        arpu,
        clv,
        revenue_lost_to_churn,
        mrr,
        fcr_rate,
        average_resolution_time,
        postpaid_fibre_renewal_rate,
        average_customer_tenure
    ],
    
    "Unit": [
        "%",
        "%",
        "INR",
        "INR",
        "INR",
        "INR",
        "%",
        "Hours",
        "%",
        "Months"
    ]
})

kpi_summary

,KPI,Value,Unit
0,Customer Churn Rate,1.29,%
1,Customer Retention Rate,98.74,%
2,ARPU,979.76,INR
3,Customer Lifetime Value,NaN,INR
4,Revenue Lost to Churn,"122,854.00",INR
5,Monthly Recurring Revenue,"14,971,268.00",INR
6,First Contact Resolution,40.41,%
7,Average Resolution Time,16.04,Hours
8,Contract Renewal Rate,59.09,%
9,Average Customer Tenure,87.10,Months


### Segment-Level KPI Analysis

In [41]:
segment_kpi = customer_kpi.merge(
    customer_arpu,
    on=["customer_id", "plan_id"],
    how="left"
)

segment_summary = (
    segment_kpi
    .groupby("customer_segment")
    .agg(
        customers=("customer_id", "nunique"),
        average_arpu=("arpu", "mean"),
        start_customers=("active_at_start", "sum"),
        churned_customers=("churned_customer", "sum"),
        end_customers=("active_at_end", "sum")
    )
    .reset_index()
)

segment_summary["churn_rate"] = (
    segment_summary["churned_customers"]
    /
    segment_summary["start_customers"]
    * 100
)

segment_summary

,customer_segment,customers,average_arpu,start_customers,churned_customers,end_customers,churn_rate
0,Enterprise,1390,"3,792.48",1221,13,1208,1.06
1,Mass,2932,249.92,2156,55,2101,2.55
2,Premium,8482,826.37,7125,67,7061,0.94
3,Value,6196,409.00,4874,63,4815,1.29


In [43]:
benchmark_summary = pd.DataFrame({
    "KPI": [
        "Customer Churn Rate",
        "First Contact Resolution",
        "Average Resolution Time",
        "Postpaid/Fibre Renewal Rate"
    ],
    
    "Actual": [
        churn_rate,
        fcr_rate,
        average_resolution_time,
        postpaid_fibre_renewal_rate
    ],
    
    "Target": [
        "< 2.10%",
        "> 78%",
        "< 24 hours",
        "> 88%"
    ]
})

benchmark_summary

,KPI,Actual,Target
0,Customer Churn Rate,1.29,< 2.10%
1,First Contact Resolution,40.41,> 78%
2,Average Resolution Time,16.04,< 24 hours
3,Postpaid/Fibre Renewal Rate,59.09,> 88%


### Save KPI Results

In [44]:
OUTPUT_DIR = DATA_DIR / "Phase3_Outputs"

OUTPUT_DIR.mkdir(
    exist_ok=True
)

kpi_summary.to_csv(
    OUTPUT_DIR / "Phase3_KPI_Summary.csv",
    index=False
)

benchmark_summary.to_csv(
    OUTPUT_DIR / "Phase3_Benchmark_Summary.csv",
    index=False
)

segment_summary.to_csv(
    OUTPUT_DIR / "Phase3_Segment_KPIs.csv",
    index=False
)

print("Phase 3 files saved successfully.")
print()
print(OUTPUT_DIR)

Phase 3 files saved successfully.

C:\Users\ASUS\Desktop\Internmo\Project - 2\Phase3_Outputs


### Cohort Retention Analysis

In [45]:
cohort_data = customers[
    [
        "customer_id",
        "acquisition_date",
        "churn_date"
    ]
].copy()

cohort_data["cohort_month"] = (
    cohort_data["acquisition_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

cohort_data.head()

,customer_id,acquisition_date,churn_date,cohort_month
0,C0000001,2026-01-15,2026-03-22,2026-01-01
1,C0000002,2014-01-01,NaT,2014-01-01
2,C0000003,2015-11-11,2015-12-22,2015-11-01
3,C0000004,NaT,NaT,NaT
4,C0000005,2014-10-20,2023-10-22,2014-10-01


### Calculate 3, 6 and 12 Month Retention

In [46]:
retention_results = []

for cohort_month, group in cohort_data.groupby("cohort_month"):
    
    if pd.isna(cohort_month):
        continue
    
    result = {
        "cohort_month": cohort_month
    }
    
    for months in [3, 6, 12]:
        
        observation_date = (
            cohort_month
            + pd.DateOffset(months=months)
        )
        
        # Only calculate if the dataset has reached
        # the required observation period.
        if observation_date > MONTH_END:
            result[f"retention_{months}m"] = np.nan
            continue
        
        total_cohort_customers = len(group)
        
        still_active = (
            group["churn_date"].isna()
            |
            (group["churn_date"] > observation_date)
        ).sum()
        
        retention = (
            still_active
            / total_cohort_customers
            * 100
        )
        
        result[f"retention_{months}m"] = retention
    
    retention_results.append(result)

cohort_retention = pd.DataFrame(
    retention_results
)

cohort_retention

,cohort_month,retention_3m,retention_6m,retention_12m
0,2014-01-01,99.85,99.66,99.31
1,2014-02-01,100.00,100.00,100.00
2,2014-03-01,100.00,100.00,100.00
3,2014-04-01,100.00,100.00,100.00
4,2014-05-01,98.92,98.92,98.92
...,...,...,...,...
142,2025-11-01,79.65,NaN,NaN
143,2025-12-01,77.14,NaN,NaN
144,2026-01-01,NaN,NaN,NaN
145,2026-02-01,NaN,NaN,NaN


### Save Cohort Analysis

In [47]:
cohort_retention.to_csv(
    OUTPUT_DIR / "Phase3_Cohort_Retention.csv",
    index=False
)

print("Cohort retention analysis saved successfully.")

Cohort retention analysis saved successfully.


### Final Management Summary

In [48]:
print("=" * 75)
print("NEXATEL CUSTOMER CHURN ANALYTICS — PHASE 3")
print("MANAGEMENT KPI SUMMARY")
print("=" * 75)

print(f"Reporting Month             : {REPORT_MONTH.strftime('%B %Y')}")
print()

print(f"Customer Churn Rate         : {churn_rate:.2f}%")
print(f"Customer Retention Rate     : {retention_rate:.2f}%")
print(f"ARPU                        : ₹{arpu:,.2f}")

if pd.notna(clv):
    print(f"Customer Lifetime Value     : ₹{clv:,.2f}")
else:
    print("Customer Lifetime Value     : Requires Gross Margin %")

print(f"Revenue Lost to Churn       : ₹{revenue_lost_to_churn:,.2f}")
print(f"Monthly Recurring Revenue   : ₹{mrr:,.2f}")
print(f"First Contact Resolution    : {fcr_rate:.2f}%")
print(f"Average Resolution Time     : {average_resolution_time:.2f} hours")
print(f"Postpaid/Fibre Renewal Rate : {postpaid_fibre_renewal_rate:.2f}%")
print(f"Average Customer Tenure     : {average_customer_tenure:.2f} months")

print("=" * 75)

NEXATEL CUSTOMER CHURN ANALYTICS — PHASE 3
MANAGEMENT KPI SUMMARY
Reporting Month             : March 2026

Customer Churn Rate         : 1.29%
Customer Retention Rate     : 98.74%
ARPU                        : ₹979.76
Customer Lifetime Value     : Requires Gross Margin %
Revenue Lost to Churn       : ₹122,854.00
Monthly Recurring Revenue   : ₹14,971,268.00
First Contact Resolution    : 40.41%
Average Resolution Time     : 16.04 hours
Postpaid/Fibre Renewal Rate : 59.09%
Average Customer Tenure     : 87.10 months
